In [1]:
!pip install -q "scikit-learn>=1.5" pandas joblib

In [2]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

PROJECT_DIR = Path(os.environ.get("RETAIL_PROJECT_DIR", Path.cwd())).resolve()
if not (PROJECT_DIR / "data").is_dir():
    PROJECT_DIR = next(
        (parent for parent in PROJECT_DIR.parents if (parent / "data").is_dir()),
        Path("/content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction")
    )
DATA_DIR = PROJECT_DIR / "data"

import os

BASE_DIR = str(PROJECT_DIR)

DATA_PATH = os.path.join(BASE_DIR, "data", "retail_cleaned.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "airflow_retraining")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Data exists:", os.path.exists(DATA_PATH))
print("Output directory:", OUTPUT_DIR)

Mounted at /content/drive
Data exists: True
Output directory: /content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction/airflow_retraining


In [3]:
# Create the reusable training script

%%writefile /content/retail_retraining.py

import os
import json
import joblib
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


def train_and_evaluate(data_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    df = pd.read_csv(data_path)
    required = {"InvoiceDate", "StockCode", "Quantity"}

    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["InvoiceDate"] = pd.to_datetime(
        df["InvoiceDate"], errors="coerce"
    )
    df = df.dropna(
        subset=["InvoiceDate", "StockCode", "Quantity"]
    )

    if "IsCancelled" in df.columns:
        df = df[df["IsCancelled"] == 0]

    df = df[df["Quantity"] > 0].copy()
    df["Date"] = df["InvoiceDate"].dt.floor("D")

    daily = (
        df.groupby(["StockCode", "Date"], as_index=False)
          .agg(DailyDemand=("Quantity", "sum"))
    )

    if daily.empty:
        raise ValueError("No valid demand records after cleaning.")

    selected_product = (
        daily.groupby("StockCode")["DailyDemand"]
             .sum()
             .idxmax()
    )

    series = (
        daily[daily["StockCode"] == selected_product]
        .set_index("Date")["DailyDemand"]
        .sort_index()
        .resample("D").sum()
    )

    full_index = pd.date_range(
        series.index.min(), series.index.max(), freq="D"
    )
    series = series.reindex(full_index, fill_value=0)
    series.index.name = "Date"

    data = pd.DataFrame({"DailyDemand": series})

    data["Lag_1"] = data["DailyDemand"].shift(1)
    data["Lag_7"] = data["DailyDemand"].shift(7)
    data["Lag_14"] = data["DailyDemand"].shift(14)

    data["RollingMean_7"] = (
        data["DailyDemand"].shift(1).rolling(7).mean()
    )
    data["RollingStd_7"] = (
        data["DailyDemand"].shift(1).rolling(7).std()
    )

    data["DayOfWeek"] = data.index.dayofweek
    data["Month"] = data.index.month

    data = data.replace([np.inf, -np.inf], np.nan).dropna()

    if len(data) < 60:
        raise ValueError(
            "Insufficient usable history for reliable evaluation."
        )

    features = [
        "Lag_1", "Lag_7", "Lag_14",
        "RollingMean_7", "RollingStd_7",
        "DayOfWeek", "Month"
    ]

    split = int(len(data) * 0.8)
    train = data.iloc[:split]
    test = data.iloc[split:]

    X_train, y_train = train[features], train["DailyDemand"]
    X_test, y_test = test[features], test["DailyDemand"]

    candidate = RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    candidate.fit(X_train, y_train)

    predictions = candidate.predict(X_test)
    candidate_mae = float(mean_absolute_error(y_test, predictions))
    candidate_rmse = float(
        np.sqrt(mean_squared_error(y_test, predictions))
    )

    model_path = os.path.join(output_dir, "champion_model.joblib")
    metrics_path = os.path.join(output_dir, "champion_metrics.json")
    candidate_path = os.path.join(output_dir, "candidate_model.joblib")
    report_path = os.path.join(output_dir, "latest_run.json")

    joblib.dump(candidate, candidate_path)

    # Promote only if there is no existing champion or the
    # candidate has a lower MAE on the same fixed holdout.
    # Existing metrics must use the same product and test dates.
    old_metrics = None
    if os.path.exists(metrics_path) and os.path.exists(model_path):
        with open(metrics_path, "r") as f:
            old_metrics = json.load(f)

    same_evaluation = (
        old_metrics is not None
        and old_metrics.get("product") == str(selected_product)
        and old_metrics.get("test_start") == str(test.index.min().date())
        and old_metrics.get("test_end") == str(test.index.max().date())
    )

    if not same_evaluation:
        promote = True
        decision = (
            "Initial champion or evaluation window changed; "
            "candidate accepted as new baseline."
        )
    elif candidate_mae < old_metrics["mae"]:
        promote = True
        decision = "Candidate MAE improved; model promoted."
    else:
        promote = False
        decision = "Candidate did not improve MAE; champion retained."

    if promote:
        joblib.dump(candidate, model_path)
        champion_metrics = {
            "product": str(selected_product),
            "mae": candidate_mae,
            "rmse": candidate_rmse,
            "test_start": str(test.index.min().date()),
            "test_end": str(test.index.max().date())
        }
        with open(metrics_path, "w") as f:
            json.dump(champion_metrics, f, indent=2)

    report = {
        "product": str(selected_product),
        "candidate_mae": candidate_mae,
        "candidate_rmse": candidate_rmse,
        "test_start": str(test.index.min().date()),
        "test_end": str(test.index.max().date()),
        "promoted": promote,
        "decision": decision
    }

    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    return report

Writing /content/retail_retraining.py


In [4]:
# Test the retraining script

import sys
sys.path.insert(0, "/content")

from retail_retraining import train_and_evaluate

result = train_and_evaluate(DATA_PATH, OUTPUT_DIR)

import json
print(json.dumps(result, indent=2))

{
  "product": "21212",
  "candidate_mae": 132.23190488967055,
  "candidate_rmse": 223.60009002692877,
  "test_start": "2010-09-29",
  "test_end": "2010-12-09",
  "promoted": true,
  "decision": "Initial champion or evaluation window changed; candidate accepted as new baseline."
}


**Evaluation note:** this is a demonstration quality gate. The candidate and champion should be compared on identical test dates and the same product, as the code checks. For production, use a fixed validation strategy, persist the exact evaluation dataset/version, and compare a new candidate against the actual saved champion rather than automatically replacing the baseline whenever the evaluation window changes.

In [5]:
# Create an Airflow DAG

%%writefile /content/retailpulse_retraining_dag.py

from datetime import datetime
import os
import sys

from airflow import DAG
from airflow.operators.python import PythonOperator

sys.path.insert(0, "/content")

from retail_retraining import train_and_evaluate

DATA_PATH = (
    os.path.join(os.environ.get("RETAIL_PROJECT_DIR", "/content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction"), "data", "retail_cleaned.csv")
)
OUTPUT_DIR = (
    os.path.join(os.environ.get("RETAIL_PROJECT_DIR", "/content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction"), "airflow_retraining")
)


def run_retraining():
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

    result = train_and_evaluate(DATA_PATH, OUTPUT_DIR)
    print(result)


def verify_report():
    report_path = os.path.join(OUTPUT_DIR, "latest_run.json")

    if not os.path.exists(report_path):
        raise FileNotFoundError("Retraining report was not created.")

    print("Retraining report verified:", report_path)


with DAG(
    dag_id="retailpulse_automated_retraining",
    description="Train, evaluate and conditionally promote retail demand model",
    start_date=datetime(2026, 1, 1),
    schedule="0 6 * * 1",
    catchup=False,
    tags=["RetailPulse", "ML", "Retraining"]
) as dag:

    train_evaluate = PythonOperator(
        task_id="train_evaluate_and_promote",
        python_callable=run_retraining
    )

    verify = PythonOperator(
        task_id="verify_training_report",
        python_callable=verify_report
    )

    train_evaluate >> verify

Writing /content/retailpulse_retraining_dag.py
